# End-to-End Ontology-Guided Verifier Evaluation

This notebook runs the released end-to-end workflow on a local trained verifier model. It starts from input sentences, applies ontology-guided preprocessing, generates candidate triples, formats verifier inputs, scores them with the selected model, and exports the accepted triples.

Pipeline:

```text
input sentences -> ontology-aware mention detection -> candidate generation -> verifier input formatting -> model scoring -> accepted triples -> JSON outputs
```

By default, the notebook uses three short test sentences and a locally available checkpoint stored under `models/`.


## 1. Setup

This section imports the required modules and resolves the repository layout used by the released code and data files.


In [1]:
from __future__ import annotations

import json
import sys
from pathlib import Path
from typing import Any

import pandas as pd


def find_repo_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for candidate in [start, *start.parents]:
        if (candidate / 'scripts').exists() and (candidate / 'datasets').exists() and (candidate / 'ontology').exists():
            return candidate
    raise FileNotFoundError('Could not locate repository root containing scripts, datasets, and ontology.')


ROOT = find_repo_root()
SCRIPTS_DIR = ROOT / 'scripts'
if str(SCRIPTS_DIR) not in sys.path:
    sys.path.insert(0, str(SCRIPTS_DIR))

import end_to_end_verifier_evaluation as evaluator  # noqa: E402
from end_to_end_verifier_evaluation import (  # noqa: E402
    format_candidate_rows,
    write_jsonl,
)

OWL_PATH = ROOT / 'ontology' / 'internal-logistics-v2.owl'
BATCH_SIZE = 32
MAX_LEN = 256

evaluator.OWL_PATH = OWL_PATH
evaluator.formatter.OWL_PATH = OWL_PATH

print('ROOT:', ROOT)
print('Scripts:', SCRIPTS_DIR)
print('Ontology:', OWL_PATH)


ROOT: C:\Users\Thomas PALLET\Documents\Code\ontology-guide-verifier-datasets-and-ontology-resource-for-logistics-knowledge-graph-construction
Scripts: C:\Users\Thomas PALLET\Documents\Code\ontology-guide-verifier-datasets-and-ontology-resource-for-logistics-knowledge-graph-construction\scripts
Ontology: C:\Users\Thomas PALLET\Documents\Code\ontology-guide-verifier-datasets-and-ontology-resource-for-logistics-knowledge-graph-construction\ontology\internal-logistics-v2.owl


## 2. Configure local model and default test sentences

This section selects a local trained checkpoint, defines three short default test sentences, and prepares the output directory used for this run.


In [2]:
MODEL_DIR_CANDIDATES = [
    ROOT / 'models' / 'main_trainable_dataset' / 'C' / 'models' / 'C_relation_entity_markers_main_trainable_dataset',
    ROOT / 'models' / 'C_relation_entity_markers',
]
SELECTED_MODEL_DIR = next((path for path in MODEL_DIR_CANDIDATES if (path / 'config.json').exists()), MODEL_DIR_CANDIDATES[0])
SELECTED_FORMAT = 'C_relation_entity_markers'
DATASET_SENTENCE_SOURCE = ROOT / 'datasets' / 'main_trainable_dataset' / 'C_relation_entity_markers' / 'test.jsonl'
OUTPUT_DIR = ROOT / 'results' / 'end_to_end_eval'
RUN_TAG = 'three_sentence_local_demo'
RUN_OUTPUT_DIR = OUTPUT_DIR / RUN_TAG

DEFAULT_TEST_SENTENCES = [
    'The demand column stores annual demand for each product.',
    'A product family should be attached when the file gives the family code.',
    'The subfamily code refines the product family with a product subfamily.',
]
MANUAL_TEXT = '\n'.join(DEFAULT_TEST_SENTENCES)

print('Selected model:', SELECTED_MODEL_DIR)
print('Model exists:', SELECTED_MODEL_DIR.exists())
print('Selected format:', SELECTED_FORMAT)
print('Default dataset sentence source:', DATASET_SENTENCE_SOURCE)
print('Default test sentences:')
for idx, sentence in enumerate(DEFAULT_TEST_SENTENCES, start=1):
    print(f'{idx}. {sentence}')
print('Run output dir:', RUN_OUTPUT_DIR)


Selected model: C:\Users\Thomas PALLET\Documents\Code\ontology-guide-verifier-datasets-and-ontology-resource-for-logistics-knowledge-graph-construction\models\main_trainable_dataset\C\models\C_relation_entity_markers_main_trainable_dataset
Model exists: True
Selected format: C_relation_entity_markers
Default dataset sentence source: C:\Users\Thomas PALLET\Documents\Code\ontology-guide-verifier-datasets-and-ontology-resource-for-logistics-knowledge-graph-construction\datasets\main_trainable_dataset\C_relation_entity_markers\test.jsonl
Default test sentences:
1. The demand column stores annual demand for each product.
2. A product family should be attached when the file gives the family code.
3. The subfamily code refines the product family with a product subfamily.
Run output dir: C:\Users\Thomas PALLET\Documents\Code\ontology-guide-verifier-datasets-and-ontology-resource-for-logistics-knowledge-graph-construction\results\end_to_end_eval\three_sentence_local_demo


## 3. Prepare the input sentences

This section defines the sentences that will be processed. By default, it uses three short test sentences, but they can be replaced manually.


In [3]:
INPUT_TEXT = MANUAL_TEXT.strip()
if not INPUT_TEXT:
    raise ValueError('INPUT_TEXT must not be empty.')

input_text = INPUT_TEXT
input_sentences = evaluator.weak.sentence_split(input_text)

print('Input sentence count:', len(input_sentences))
for idx, sentence in enumerate(input_sentences, start=1):
    print(f'{idx}. {sentence}')


Input sentence count: 3
1. The demand column stores annual demand for each product.
2. A product family should be attached when the file gives the family code.
3. The subfamily code refines the product family with a product subfamily.


## 4. Generate ontology-guided candidates

This section applies ontology-aware mention detection and candidate generation to the selected input sentence.


In [4]:
raw_candidate_rows = evaluator.build_candidate_rows_from_text(input_text)
candidate_rows = format_candidate_rows(raw_candidate_rows)
sentence_summaries = evaluator.candidate_rows_to_sentence_summary(candidate_rows)

RUN_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
write_jsonl(RUN_OUTPUT_DIR / 'all_generated_candidates.jsonl', candidate_rows)

candidate_df = pd.DataFrame(candidate_rows)
sentence_summary_df = pd.DataFrame(sentence_summaries)

print('Generated candidates:', len(candidate_rows))
print('Sentences with candidates:', len(sentence_summaries))
print('Candidate export:', RUN_OUTPUT_DIR / 'all_generated_candidates.jsonl')

if not sentence_summary_df.empty:
    display(sentence_summary_df[['sentence_id', 'sentence', 'candidate_count', 'candidate_relations_text']])
else:
    print('No candidates generated for the input sentence.')


Generated candidates: 4
Sentences with candidates: 3
Candidate export: C:\Users\Thomas PALLET\Documents\Code\ontology-guide-verifier-datasets-and-ontology-resource-for-logistics-knowledge-graph-construction\results\end_to_end_eval\three_sentence_local_demo\all_generated_candidates.jsonl


,sentence_id,sentence,candidate_count,candidate_relations_text
0,1,The demand column stores annual demand for eac...,1,annualDemand
1,2,A product family should be attached when the f...,1,belongsToFamily
2,3,The subfamily code refines the product family ...,2,"belongsToFamily, belongsToSubfamily"


## 5. Inspect verifier inputs

This section shows how one generated candidate is represented in the retained verifier input formats.


In [5]:
EXAMPLE_INDEX = 0

if not candidate_rows:
    print('No candidates generated.')
else:
    row = candidate_rows[EXAMPLE_INDEX]
    print('Candidate:', row['candidate_id'])
    print('Sentence:', row['sentence'])
    print('Subject / predicate / object:', row['subject_short'], row['candidate_relation_short'], row['object_short'])
    for format_name in ['A_sentence_only', 'B_relation_marker', 'C_relation_entity_markers', 'D_ontology_context', 'F_full_ontology_context']:
        print('\n' + '=' * 100)
        print(format_name)
        print('=' * 100)
        print(row['input_formats'][format_name])


Candidate: s001_c001
Sentence: The demand column stores annual demand for each product.
Subject / predicate / object: Product annualDemand LiteralValue

A_sentence_only
The demand column stores annual demand for each product.

B_relation_marker
[REL] annualDemand [/REL] The demand column stores annual demand for each product.

C_relation_entity_markers
[REL] annualDemand [/REL] The demand column stores [E1]annual demand[/E1] for each [E2]product[/E2].

D_ontology_context
[SUBJ] Product [/SUBJ] [SUBJ_TYPE] Product [/SUBJ_TYPE] [REL] annualDemand [/REL] [OBJ] LiteralValue [/OBJ] [OBJ_TYPE] LiteralValue [/OBJ_TYPE] [DOMAIN] Product [/DOMAIN] [RANGE] integer [/RANGE] The demand column stores [E1]annual demand[/E1] for each [E2]product[/E2].

F_full_ontology_context
[SUBJ] Product [/SUBJ] [SUBJ_TYPE] Product [/SUBJ_TYPE] [REL] annualDemand [/REL] [OBJ] LiteralValue [/OBJ] [OBJ_TYPE] LiteralValue [/OBJ_TYPE] [DOMAIN] Product [/DOMAIN] [RANGE] integer [/RANGE] [REL_TYPE] datatype_property [/R

## 6. Score candidates with the selected model

This section applies the selected local verifier checkpoint to the generated candidates and computes the proportion of candidates retained as `VALID`.


In [6]:
if not (SELECTED_MODEL_DIR / 'config.json').exists():
    raise FileNotFoundError(f'Missing local model checkpoint: {SELECTED_MODEL_DIR}')

model_inputs = evaluator.prepare_inputs(SELECTED_FORMAT, candidate_rows)
predictions = evaluator.predict_texts(
    SELECTED_MODEL_DIR,
    model_inputs,
    batch_size=BATCH_SIZE,
    max_len=MAX_LEN,
)
enriched_predictions = evaluator.enrich_rows_with_predictions(candidate_rows, predictions)
accepted_rows = [row for row in enriched_predictions if row['prediction'] == 'VALID']
accepted_triples = evaluator.triples_from_predictions(candidate_rows, predictions)
retained_ratio = evaluator.accepted_ratio(candidate_rows, predictions)

prediction_df = pd.DataFrame(enriched_predictions)
accepted_df = pd.DataFrame(accepted_triples)

write_jsonl(RUN_OUTPUT_DIR / 'candidate_predictions.jsonl', enriched_predictions)
write_jsonl(RUN_OUTPUT_DIR / 'accepted_triples.jsonl', accepted_triples)

print('Selected model:', SELECTED_MODEL_DIR)
print('Selected format:', SELECTED_FORMAT)
print('Candidates scored:', len(candidate_rows))
print('Accepted triples:', len(accepted_triples))
print(f'Retention rate: {retained_ratio:.1%}')


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Selected model: C:\Users\Thomas PALLET\Documents\Code\ontology-guide-verifier-datasets-and-ontology-resource-for-logistics-knowledge-graph-construction\models\main_trainable_dataset\C\models\C_relation_entity_markers_main_trainable_dataset
Selected format: C_relation_entity_markers
Candidates scored: 4
Accepted triples: 4
Retention rate: 100.0%


## 7. Review candidate predictions

This section lists the generated candidates together with their predicted labels and `VALID` scores.


In [7]:
prediction_cols = [
    'sentence_id', 'candidate_id', 'subject_short', 'candidate_relation_short', 'object_short',
    'prediction', 'score_valid', 'score_invalid', 'domain_range_valid', 'unit_compatible',
    'ontology_validation_flag', 'sentence'
]
existing_prediction_cols = [col for col in prediction_cols if col in prediction_df.columns]
if not prediction_df.empty:
    display(prediction_df[existing_prediction_cols].sort_values(['sentence_id', 'score_valid'], ascending=[True, False]))
else:
    print('No predictions available.')


,sentence_id,candidate_id,subject_short,candidate_relation_short,object_short,prediction,score_valid,score_invalid,domain_range_valid,unit_compatible,ontology_validation_flag,sentence
0,1,s001_c001,Product,annualDemand,LiteralValue,VALID,0.970644,0.029356,True,None,STRUCTURALLY_VALID,The demand column stores annual demand for eac...
1,2,s002_c001,Product,belongsToFamily,ProductFamily,VALID,0.874542,0.125458,None,None,NOT_APPLICABLE,A product family should be attached when the f...
3,3,s003_c002,Product,belongsToSubfamily,ProductSubfamily,VALID,0.974502,0.025498,True,None,STRUCTURALLY_VALID,The subfamily code refines the product family ...
2,3,s003_c001,Product,belongsToFamily,ProductFamily,VALID,0.966568,0.033432,None,None,NOT_APPLICABLE,The subfamily code refines the product family ...


## 8. Inspect accepted triples

This section focuses on the candidates retained by the verifier and summarizes the accepted ontology-grounded triples.


In [8]:
if not accepted_df.empty:
    display(accepted_df[['sentence_id', 'subject', 'predicate', 'object', 'score_valid', 'sentence']])
else:
    print('No triples were accepted by the selected model.')

print('Processing summary')
print('- ontology parsed:', OWL_PATH)
print('- sentence preprocessing executed:', len(input_sentences), 'sentence(s)')
print('- candidate generation executed:', len(candidate_rows), 'candidate(s)')
print('- model scoring executed with:', SELECTED_MODEL_DIR.name)
print(f'- candidate retention after verification: {len(accepted_triples)}/{len(candidate_rows)} ({retained_ratio:.1%})')
print('\nValidation flag guide')
print('- STRUCTURALLY_VALID: the accepted candidate is also supported by the ontology-level structural checks used in preprocessing.')
print('- NOT_APPLICABLE: the candidate was accepted by the verifier, but the preprocessing stage did not attach an explicit structural validation flag for that relation type.')
print('- STRUCTURALLY_INVALID: a candidate with this flag should be treated cautiously even if the model assigns a high VALID score.')


,sentence_id,subject,predicate,object,score_valid,sentence
0,1,Product,annualDemand,LiteralValue,0.970644,The demand column stores annual demand for eac...
1,2,Product,belongsToFamily,ProductFamily,0.874542,A product family should be attached when the f...
2,3,Product,belongsToFamily,ProductFamily,0.966568,The subfamily code refines the product family ...
3,3,Product,belongsToSubfamily,ProductSubfamily,0.974502,The subfamily code refines the product family ...


Processing summary
- ontology parsed: C:\Users\Thomas PALLET\Documents\Code\ontology-guide-verifier-datasets-and-ontology-resource-for-logistics-knowledge-graph-construction\ontology\internal-logistics-v2.owl
- sentence preprocessing executed: 3 sentence(s)
- candidate generation executed: 4 candidate(s)
- model scoring executed with: C_relation_entity_markers_main_trainable_dataset
- candidate retention after verification: 4/4 (100.0%)

Validation flag guide
- STRUCTURALLY_VALID: the accepted candidate is also supported by the ontology-level structural checks used in preprocessing.
- NOT_APPLICABLE: the candidate was accepted by the verifier, but the preprocessing stage did not attach an explicit structural validation flag for that relation type.
- STRUCTURALLY_INVALID: a candidate with this flag should be treated cautiously even if the model assigns a high VALID score.


## 9. Saved files

This section lists the files written for the current local end-to-end run.


In [9]:
accepted_path = RUN_OUTPUT_DIR / 'accepted_triples.jsonl'
predictions_path = RUN_OUTPUT_DIR / 'candidate_predictions.jsonl'
candidates_path = RUN_OUTPUT_DIR / 'all_generated_candidates.jsonl'

print('Run directory:', RUN_OUTPUT_DIR)
print('Generated candidates:', candidates_path, 'exists=', candidates_path.exists())
print('Candidate predictions:', predictions_path, 'exists=', predictions_path.exists())
print('Accepted triples:', accepted_path, 'exists=', accepted_path.exists())


Run directory: C:\Users\Thomas PALLET\Documents\Code\ontology-guide-verifier-datasets-and-ontology-resource-for-logistics-knowledge-graph-construction\results\end_to_end_eval\three_sentence_local_demo
Generated candidates: C:\Users\Thomas PALLET\Documents\Code\ontology-guide-verifier-datasets-and-ontology-resource-for-logistics-knowledge-graph-construction\results\end_to_end_eval\three_sentence_local_demo\all_generated_candidates.jsonl exists= True
Candidate predictions: C:\Users\Thomas PALLET\Documents\Code\ontology-guide-verifier-datasets-and-ontology-resource-for-logistics-knowledge-graph-construction\results\end_to_end_eval\three_sentence_local_demo\candidate_predictions.jsonl exists= True
Accepted triples: C:\Users\Thomas PALLET\Documents\Code\ontology-guide-verifier-datasets-and-ontology-resource-for-logistics-knowledge-graph-construction\results\end_to_end_eval\three_sentence_local_demo\accepted_triples.jsonl exists= True


The current run has completed. The notebook has parsed the ontology, generated candidate triples from the input sentences, scored them with the selected local verifier model, and exported both the full candidate predictions and the accepted triples.
